# Evaluation — Ground Truth + Metrics

This notebook answers: **are TF-IDF embeddings actually good for travel retrieval?**

We build two types of ground truth and measure four systems against them.

### Two ground-truth strategies

| Strategy | How | Measures |
|----------|-----|----------|
| **Label-based** | Human-defined rules: same category = relevant, same country = partial | *Embedding quality* — does the embedding capture the right semantic similarity? |
| **Oracle (brute-force)** | Exact cosine search over the raw embeddings | *Approximate search quality* — does HNSW find what the embedding says is similar? |

### Metrics
- **Recall@K** — of the K truly relevant results, how many did we find?
- **Precision@K** — of the K results we returned, how many are relevant?
- **MRR** (Mean Reciprocal Rank) — is the first relevant result at rank 1, 2, 3 …?
- **NDCG@K** — weighted by rank position; finding relevant results early scores higher

## 0. Setup — re-run everything from v1 and v2

In [1]:
%pip install numpy scikit-learn sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import math, random, heapq
import numpy as np
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer

PLACES = [
    {"name": "Eiffel Tower",           "country": "France",         "description": "Iconic iron lattice tower on the Champ de Mars in Paris. Symbol of romance and French engineering. Stunning views of the city skyline.",                                                                    "best_time": "April–June",      "category": "landmark"},
    {"name": "Mont Saint-Michel",       "country": "France",         "description": "Medieval abbey perched on a tidal island in Normandy. Surrounded by vast sandflats and tidal waters. One of France's most visited monuments.",                                                           "best_time": "May–September",   "category": "heritage"},
    {"name": "Palace of Versailles",    "country": "France",         "description": "Opulent royal château with baroque gardens outside Paris. Former residence of Louis XIV. UNESCO World Heritage Site.",                                                                                    "best_time": "April–October",   "category": "heritage"},
    {"name": "French Riviera",          "country": "France",         "description": "Mediterranean coastline with glamorous beach resorts, Nice, Cannes and Monaco. Crystal blue waters and luxury yachts. Vibrant nightlife.",                                                              "best_time": "June–August",     "category": "beach"},
    {"name": "Loire Valley",            "country": "France",         "description": "Valley of châteaux along the Loire river. Renaissance architecture, vineyards and cycling paths through lush French countryside.",                                                                       "best_time": "May–October",     "category": "nature"},
    {"name": "Colosseum",               "country": "Italy",          "description": "Ancient Roman amphitheatre in the heart of Rome. Largest amphitheatre ever built. Iconic symbol of Imperial Rome and gladiatorial combat.",                                                              "best_time": "March–May",       "category": "heritage"},
    {"name": "Amalfi Coast",            "country": "Italy",          "description": "Dramatic cliffside villages along the Tyrrhenian Sea south of Naples. Pastel-coloured houses, limoncello, and turquoise coves.",                                                                        "best_time": "May–October",     "category": "beach"},
    {"name": "Venice Canals",           "country": "Italy",          "description": "City built on water with gondolas navigating narrow canals. St Mark's Basilica and Doge's Palace reflect Byzantine grandeur. Carnival in February.",                                                    "best_time": "April–June",      "category": "landmark"},
    {"name": "Tuscany Hills",           "country": "Italy",          "description": "Rolling hills with cypress trees, vineyards and medieval hilltop towns. Chianti wine country. Siena and San Gimignano nearby.",                                                                         "best_time": "April–October",   "category": "nature"},
    {"name": "Pompeii",                 "country": "Italy",          "description": "Ancient Roman city preserved under volcanic ash from Mount Vesuvius eruption in AD79. Extraordinary window into Roman daily life.",                                                                     "best_time": "March–May",       "category": "heritage"},
    {"name": "Cinque Terre",            "country": "Italy",          "description": "Five colourful fishing villages clinging to cliffs on the Ligurian coast. Hiking trails with dramatic sea views. Fresh seafood and pesto.",                                                            "best_time": "May–September",   "category": "beach"},
    {"name": "Dolomites",               "country": "Italy",          "description": "Alpine mountain range in northeastern Italy with dramatic jagged peaks. World-class skiing in winter and hiking in summer. UNESCO heritage.",                                                            "best_time": "June–September",  "category": "nature"},
    {"name": "Sagrada Familia",         "country": "Spain",          "description": "Gaudí's unfinished basilica in Barcelona. Extraordinary organic architecture blending Gothic and Art Nouveau. Under construction since 1882.",                                                          "best_time": "March–May",       "category": "landmark"},
    {"name": "Alhambra",                "country": "Spain",          "description": "Moorish palace complex in Granada with intricate Islamic architecture. Stunning views over the Sierra Nevada. UNESCO World Heritage Site.",                                                              "best_time": "March–June",      "category": "heritage"},
    {"name": "Ibiza",                   "country": "Spain",          "description": "Island in the Balearics famous for nightlife and electronic music. Also has quiet coves, old town Dalt Vila and crystal-clear Mediterranean waters.",                                                  "best_time": "June–September",  "category": "beach"},
    {"name": "Park Güell",              "country": "Spain",          "description": "Gaudí's mosaic-covered public park on a hill above Barcelona. Colourful tiled benches and organic structures with sweeping city views.",                                                              "best_time": "April–October",   "category": "landmark"},
    {"name": "Camino de Santiago",      "country": "Spain",          "description": "Ancient pilgrimage route across northern Spain ending at Santiago Cathedral. 800km trail through diverse landscapes, villages and vineyards.",                                                          "best_time": "May–September",   "category": "nature"},
    {"name": "Santorini",               "country": "Greece",         "description": "Volcanic island with iconic white-washed buildings and blue-domed churches. Dramatic caldera views and stunning sunsets from Oia.",                                                                     "best_time": "June–September",  "category": "beach"},
    {"name": "Acropolis of Athens",     "country": "Greece",         "description": "Ancient citadel above Athens containing the Parthenon temple. Masterpiece of ancient Greek architecture. UNESCO World Heritage Site.",                                                                 "best_time": "April–May",       "category": "heritage"},
    {"name": "Meteora",                 "country": "Greece",         "description": "Eastern Orthodox monasteries perched on natural sandstone pillars in central Greece. Otherworldly landscape with dramatic rock formations.",                                                            "best_time": "April–June",      "category": "heritage"},
    {"name": "Mykonos",                 "country": "Greece",         "description": "Cyclades island renowned for cosmopolitan beach clubs, windmills and vibrant LGBTQ+ nightlife. Picturesque Little Venice waterfront.",                                                                "best_time": "June–September",  "category": "beach"},
    {"name": "Crete",                   "country": "Greece",         "description": "Largest Greek island with Minoan ruins at Knossos, Samaria Gorge hikes and diverse beaches from pink sand to palm-lined coves.",                                                                     "best_time": "May–October",     "category": "nature"},
    {"name": "Neuschwanstein Castle",   "country": "Germany",        "description": "Fairytale castle commissioned by King Ludwig II in the Bavarian Alps. Inspiration for Disney's Sleeping Beauty castle. Stunning Alpine backdrop.",                                                    "best_time": "May–September",   "category": "landmark"},
    {"name": "Berlin Wall Memorial",    "country": "Germany",        "description": "Memorial site along the former Berlin Wall with preserved segments and documentation centre. Powerful reminder of Cold War division.",                                                                  "best_time": "Year-round",      "category": "heritage"},
    {"name": "Black Forest",            "country": "Germany",        "description": "Dense forested mountain range in southwest Germany. Cuckoo clocks, kirsch cake, thermal baths and hiking through dense dark spruce woods.",                                                            "best_time": "May–October",     "category": "nature"},
    {"name": "Rhine Valley",            "country": "Germany",        "description": "Romantic Rhine Gorge with medieval castles on hillsides above the river. Wine villages and boat cruises through UNESCO protected landscape.",                                                          "best_time": "May–October",     "category": "nature"},
    {"name": "Vienna Schönbrunn Palace","country": "Austria",        "description": "Baroque imperial palace and gardens of the Habsburg dynasty in Vienna. 1441 rooms and a hilltop Gloriette with panoramic city views.",                                                               "best_time": "April–October",   "category": "heritage"},
    {"name": "Hallstatt",               "country": "Austria",        "description": "Picturesque lakeside village in the Salzkammergut mountains. Steep salt mine history, wooden architecture and mirror-like Alpine lake.",                                                              "best_time": "May–September",   "category": "nature"},
    {"name": "Matterhorn",              "country": "Switzerland",    "description": "Iconic pyramid-shaped peak on the Swiss-Italian border near Zermatt. One of the highest peaks in the Alps. World-class skiing resort.",                                                               "best_time": "June–September",  "category": "nature"},
    {"name": "Lake Geneva",             "country": "Switzerland",    "description": "Large crescent-shaped lake on the Swiss-French border. Vineyards on terrace hillsides, medieval Chillon Castle and the Jet d'Eau fountain.",                                                         "best_time": "May–September",   "category": "nature"},
    {"name": "Lisbon Historic Trams",   "country": "Portugal",       "description": "Iconic yellow trams winding through steep Alfama neighbourhood cobblestone streets. Fado music, pastéis de nata and Moorish castle views.",                                                          "best_time": "March–May",       "category": "landmark"},
    {"name": "Algarve Beaches",         "country": "Portugal",       "description": "Atlantic coastline with golden limestone cliffs, sea caves and turquoise lagoons. Surfing, seafood restaurants and historic fishing villages.",                                                       "best_time": "June–September",  "category": "beach"},
    {"name": "Sintra Palaces",          "country": "Portugal",       "description": "Fairytale town in forested hills near Lisbon with Romantic-era palaces including Pena Palace and the Moorish Castle ruins.",                                                                         "best_time": "March–October",   "category": "heritage"},
    {"name": "Prague Old Town",         "country": "Czech Republic", "description": "Medieval old town with astronomical clock, Gothic churches and Baroque palaces around cobblestone squares. Bohemian beer culture.",                                                                   "best_time": "April–October",   "category": "heritage"},
    {"name": "Český Krumlov",           "country": "Czech Republic", "description": "Tiny UNESCO heritage town in Bohemia with a massive castle above a river bend. Baroque theatre, rafting and atmospheric cobblestone lanes.",                                                          "best_time": "May–September",   "category": "heritage"},
    {"name": "Keukenhof Gardens",       "country": "Netherlands",    "description": "World's largest flower garden near Amsterdam with 7 million tulip bulbs in bloom. Open only in spring. Windmill and Dutch pastoral landscape.",                                                       "best_time": "March–May",       "category": "nature"},
    {"name": "Amsterdam Canals",        "country": "Netherlands",    "description": "17th-century canal ring with narrow gabled houses, houseboats and cycling culture. Van Gogh Museum, Rijksmuseum and Anne Frank House nearby.",                                                       "best_time": "April–September", "category": "landmark"},
    {"name": "Norwegian Fjords",        "country": "Norway",         "description": "Spectacular glacially carved inlets with waterfalls dropping from steep cliffs. Cruise or kayak through Geirangerfjord and Nærøyfjord UNESCO sites.",                                               "best_time": "May–September",   "category": "nature"},
    {"name": "Northern Lights Iceland", "country": "Iceland",        "description": "Aurora borealis dancing across Arctic skies. Hot springs, geysers, black sand beaches and midnight sun in the land of fire and ice.",                                                               "best_time": "September–March", "category": "nature"},
    {"name": "Stockholm Gamla Stan",    "country": "Sweden",         "description": "Stockholm's medieval old town on a small island. Colourful 17th-century buildings, Royal Palace, Nobel Museum and winding cobblestone alleys.",                                                     "best_time": "June–August",     "category": "heritage"},
    {"name": "Budapest Thermal Baths",  "country": "Hungary",        "description": "City built on natural hot springs with grand Ottoman and Art Nouveau bath houses. Széchenyi and Gellért baths are the most famous.",                                                                 "best_time": "Year-round",      "category": "landmark"},
    {"name": "Dubrovnik Old City",      "country": "Croatia",        "description": "Walled medieval city on the Adriatic coast. Walk the city walls above terracotta rooftops and turquoise sea. Game of Thrones filming location.",                                                    "best_time": "May–June",        "category": "heritage"},
    {"name": "Plitvice Lakes",          "country": "Croatia",        "description": "Cascading turquoise lakes and waterfalls in a forested canyon. UNESCO national park with wooden boardwalks above crystal-clear mineral waters.",                                                    "best_time": "April–October",   "category": "nature"},
    {"name": "Transylvania",            "country": "Romania",        "description": "Region of medieval castles, fortified churches and Carpathian mountain forests. Bran Castle, Dracula legend and authentic rural village life.",                                                      "best_time": "May–September",   "category": "heritage"},
    {"name": "Scottish Highlands",      "country": "Scotland",       "description": "Wild landscape of glens, lochs and dramatic mountains. Loch Ness, Glencoe valley, Eilean Donan castle and whisky distillery tours.",                                                               "best_time": "May–September",   "category": "nature"},
    {"name": "Cliffs of Moher",         "country": "Ireland",        "description": "Dramatic 214-metre sea cliffs on the Atlantic coast of County Clare. Puffin colonies, crashing waves and clear days with Aran Islands views.",                                                     "best_time": "April–September", "category": "nature"},
    {"name": "Stonehenge",              "country": "England",        "description": "Prehistoric stone circle monument in Wiltshire. Built 3000–1500 BC. Alignment with summer solstice sunrise. UNESCO World Heritage Site.",                                                            "best_time": "April–September", "category": "heritage"},
]

DOCS = [f"{p['name']} {p['country']} {p['category']} {p['description']}" for p in PLACES]
N    = len(PLACES)
IDX  = {p['name']: i for i, p in enumerate(PLACES)}  # name → index
print(f"Loaded {N} places")

/Users/harshita/Documents/github_repos/trip_buddy/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 47 places


In [3]:
# ── Build embeddings ─────────────────────────────────────────────────────────
tfidf_vec   = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
sparse_embs = normalize(tfidf_vec.fit_transform(DOCS).toarray(), norm='l2')

sent_model  = SentenceTransformer("all-MiniLM-L6-v2")
dense_embs  = sent_model.encode(DOCS, normalize_embeddings=True, show_progress_bar=True)

print(f"Sparse: {sparse_embs.shape}  |  Dense: {dense_embs.shape}")

Batches: 100%|██████████| 2/2 [00:00<00:00, 17.29it/s]

Sparse: (47, 1548)  |  Dense: (47, 384)


In [4]:
# ── HNSW class (same as v1/v2) ────────────────────────────────────────────────
class HNSWIndex:
    def __init__(self, M=16, M0=None, ef_construction=200, ef_search=50):
        self.M, self.M0 = M, (M0 or 2*M)
        self.ef_construction, self.ef_search = ef_construction, ef_search
        self.m_L = 1.0 / math.log(M) if M > 1 else 1.0
        self.vectors, self.labels = [], []
        self.max_layer, self.entry_point = -1, None
        self.graphs = defaultdict(lambda: defaultdict(set))

    def _sim(self, a, b): return float(np.dot(a, b))
    def _random_level(self): return int(-math.log(random.random()) * self.m_L)

    def _search_layer(self, q, entry_ids, ef, layer):
        visited = set(entry_ids)
        cands, found = [], []
        for e in entry_ids:
            s = self._sim(q, self.vectors[e])
            heapq.heappush(cands, (-s, e)); heapq.heappush(found, (-s, e))
        while cands:
            ns, c = heapq.heappop(cands)
            if -ns < -found[0][0]: break
            for nb in self.graphs[layer][c]:
                if nb in visited: continue
                visited.add(nb); sn = self._sim(q, self.vectors[nb])
                if len(found) < ef or sn > -found[0][0]:
                    heapq.heappush(cands, (-sn, nb)); heapq.heappush(found, (-sn, nb))
                    if len(found) > ef: heapq.heappop(found)
        return [(-ns, nid) for ns, nid in found]

    def _select_neighbors(self, q, cands, M):
        sel = []
        for sc, c in sorted(cands, key=lambda x: -x[0]):
            if len(sel) >= M: break
            if all(self._sim(self.vectors[c], self.vectors[s]) <= sc for _, s in sel):
                sel.append((sc, c))
        return sel

    def insert(self, vec, label=None):
        nid = len(self.vectors); self.vectors.append(vec); self.labels.append(label)
        lv = self._random_level(); eids = [self.entry_point] if self.entry_point is not None else []
        if self.entry_point is not None:
            for l in range(self.max_layer, lv, -1):
                eids = [self._search_layer(vec, eids, 1, l)[0][1]]
        for l in range(min(lv, self.max_layer), -1, -1):
            if not eids: break
            cands = self._search_layer(vec, eids, self.ef_construction, l)
            Ml = self.M0 if l == 0 else self.M
            nbrs = self._select_neighbors(vec, cands, Ml)
            for _, n in nbrs:
                self.graphs[l][nid].add(n); self.graphs[l][n].add(nid)
                if len(self.graphs[l][n]) > Ml:
                    c2 = [(self._sim(self.vectors[n], self.vectors[x]), x) for x in self.graphs[l][n]]
                    self.graphs[l][n] = {x for _, x in self._select_neighbors(self.vectors[n], c2, Ml)}
            eids = [n for _, n in nbrs]
        if lv > self.max_layer: self.max_layer = lv; self.entry_point = nid
        return nid

    def search(self, q, k=5):
        if self.entry_point is None: return []
        eids = [self.entry_point]
        for l in range(self.max_layer, 0, -1):
            eids = [self._search_layer(q, eids, 1, l)[0][1]]
        res = self._search_layer(q, eids, self.ef_search, 0)
        res.sort(key=lambda x: -x[0])
        return [(s, self.labels[n]) for s, n in res[:k]]

# Build both indexes
random.seed(42)
sparse_idx = HNSWIndex(M=8, M0=16, ef_construction=100, ef_search=50)
for v, p in zip(sparse_embs, PLACES): sparse_idx.insert(v, label=p)

random.seed(42)
dense_idx = HNSWIndex(M=8, M0=16, ef_construction=100, ef_search=50)
for v, p in zip(dense_embs, PLACES): dense_idx.insert(v, label=p)

print("Both HNSW indexes built.")

Both HNSW indexes built.


---
## 1. Ground truth — label-based relevance

### Why we need this
Brute-force similarity over the TF-IDF embeddings would just confirm that TF-IDF finds what TF-IDF considers similar — a circular evaluation. We need an **independent** signal.

### Relevance scoring rules

| Condition | Relevance score |
|-----------|----------------|
| Same `category` AND same `country` | 2 (highly relevant) |
| Same `category` only | 1 (relevant) |
| Different category, same country | 0.5 (weakly relevant) |
| Neither | 0 (not relevant) |

For binary metrics (Recall, Precision) we threshold: score ≥ 1 → relevant.

In [5]:
def label_relevance(a: dict, b: dict) -> float:
    """Graded relevance score between two places based on category and country."""
    if a['name'] == b['name']:
        return 0.0  # exclude self from ground truth
    same_cat     = a['category'] == b['category']
    same_country = a['country']  == b['country']
    if same_cat and same_country: return 2.0
    if same_cat:                  return 1.0
    if same_country:              return 0.5
    return 0.0


def label_ground_truth(query_place: dict, k: int,
                        threshold: float = 1.0) -> list[int]:
    """
    Returns a sorted list of place indexes that are relevant to query_place,
    ordered by relevance score descending, capped at k.
    """
    scored = [
        (label_relevance(query_place, p), i)
        for i, p in enumerate(PLACES)
        if label_relevance(query_place, p) >= threshold
    ]
    scored.sort(key=lambda x: -x[0])
    return [i for _, i in scored[:k]]


# Preview ground truth for a few places
for name in ["Santorini", "Colosseum", "Scottish Highlands"]:
    q  = PLACES[IDX[name]]
    gt = label_ground_truth(q, k=10)
    gt_names = [PLACES[i]['name'] for i in gt]
    print(f"\nGround truth for '{name}' (category={q['category']}, country={q['country']}):")
    for i, n in zip(gt, gt_names):
        rel = label_relevance(q, PLACES[i])
        print(f"  [{rel:.1f}] {n} ({PLACES[i]['category']}, {PLACES[i]['country']})")


Ground truth for 'Santorini' (category=beach, country=Greece):
  [2.0] Mykonos (beach, Greece)
  [1.0] French Riviera (beach, France)
  [1.0] Amalfi Coast (beach, Italy)
  [1.0] Cinque Terre (beach, Italy)
  [1.0] Ibiza (beach, Spain)
  [1.0] Algarve Beaches (beach, Portugal)

Ground truth for 'Colosseum' (category=heritage, country=Italy):
  [2.0] Pompeii (heritage, Italy)
  [1.0] Mont Saint-Michel (heritage, France)
  [1.0] Palace of Versailles (heritage, France)
  [1.0] Alhambra (heritage, Spain)
  [1.0] Acropolis of Athens (heritage, Greece)
  [1.0] Meteora (heritage, Greece)
  [1.0] Berlin Wall Memorial (heritage, Germany)
  [1.0] Vienna Schönbrunn Palace (heritage, Austria)
  [1.0] Sintra Palaces (heritage, Portugal)
  [1.0] Prague Old Town (heritage, Czech Republic)

Ground truth for 'Scottish Highlands' (category=nature, country=Scotland):
  [1.0] Loire Valley (nature, France)
  [1.0] Tuscany Hills (nature, Italy)
  [1.0] Dolomites (nature, Italy)
  [1.0] Camino de Santiago (n

---
## 2. Ground truth — oracle (brute-force exact search)

The oracle answers: *does HNSW find what the embedding thinks is similar?*  
This separates the **approximate search error** (HNSW graph quality) from the **embedding error** (is the embedding even capturing the right thing?).

In [6]:
def oracle_top_k(emb_matrix: np.ndarray, query_idx: int, k: int) -> list[int]:
    """
    Exact nearest neighbours for place at query_idx using full matrix dot product.
    Excludes the query itself (rank 0 is always self).
    """
    sims = emb_matrix @ emb_matrix[query_idx]   # cosine sims for all N places
    sims[query_idx] = -1.0                       # exclude self
    return list(np.argsort(sims)[::-1][:k])


# Show oracle top-5 for Santorini under both embeddings
for name in ["Santorini", "Colosseum", "Scottish Highlands"]:
    qi = IDX[name]
    sparse_nn = oracle_top_k(sparse_embs, qi, k=5)
    dense_nn  = oracle_top_k(dense_embs,  qi, k=5)
    print(f"\nOracle top-5 for '{name}':")
    print(f"  {'Sparse (TF-IDF)':<30}  {'Dense (SentTrans)'}")
    for s, d in zip(sparse_nn, dense_nn):
        print(f"  {PLACES[s]['name']:<30}  {PLACES[d]['name']}")


Oracle top-5 for 'Santorini':
  Sparse (TF-IDF)                 Dense (SentTrans)
  Mykonos                         Meteora
  Crete                           Crete
  French Riviera                  Mykonos
  Meteora                         French Riviera
  Eiffel Tower                    Amalfi Coast

Oracle top-5 for 'Colosseum':
  Sparse (TF-IDF)                 Dense (SentTrans)
  Pompeii                         Pompeii
  Eiffel Tower                    Venice Canals
  Acropolis of Athens             Acropolis of Athens
  Matterhorn                      Budapest Thermal Baths
  Vienna Schönbrunn Palace        Berlin Wall Memorial

Oracle top-5 for 'Scottish Highlands':
  Sparse (TF-IDF)                 Dense (SentTrans)
  Loire Valley                    Loire Valley
  Rhine Valley                    Rhine Valley
  Lake Geneva                     Black Forest
  Hallstatt                       Tuscany Hills
  Cliffs of Moher                 Hallstatt


---
## 3. Metric implementations

All four metrics accept a **ranked list of returned items** and a **set of relevant item IDs**.

In [7]:
def recall_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    """
    Recall@K = |retrieved[:k] ∩ relevant| / min(k, |relevant|)
    Denominator is capped at k so scores don't exceed 1.0 when |relevant| < k.
    """
    if not relevant_ids:
        return 0.0
    hits = len(set(retrieved_ids[:k]) & relevant_ids)
    return hits / min(k, len(relevant_ids))


def precision_at_k(retrieved_ids: list[int], relevant_ids: set[int], k: int) -> float:
    """
    Precision@K = |retrieved[:k] ∩ relevant| / k
    """
    if not retrieved_ids:
        return 0.0
    hits = len(set(retrieved_ids[:k]) & relevant_ids)
    return hits / k


def mrr(retrieved_ids: list[int], relevant_ids: set[int]) -> float:
    """
    Mean Reciprocal Rank — reciprocal of the rank of the FIRST relevant result.
    Returns 0 if no relevant result found in the list.
    """
    for rank, rid in enumerate(retrieved_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved_ids: list[int], relevance_scores: dict[int, float], k: int) -> float:
    """
    NDCG@K — rewards finding highly-relevant results early.

    DCG  = sum( rel(i) / log2(rank+1) )  for rank 1..k
    IDCG = DCG of the ideal (perfect) ranking
    NDCG = DCG / IDCG

    relevance_scores: {place_index: graded_score}  (0.0 if not relevant)
    """
    dcg = sum(
        relevance_scores.get(rid, 0.0) / math.log2(rank + 1)
        for rank, rid in enumerate(retrieved_ids[:k], start=1)
    )
    ideal_rels = sorted(relevance_scores.values(), reverse=True)[:k]
    idcg = sum(
        rel / math.log2(rank + 1)
        for rank, rel in enumerate(ideal_rels, start=1)
    )
    return dcg / idcg if idcg > 0 else 0.0


print("Metric functions defined.")

Metric functions defined.


---
## 4. Spot-check: why does Santorini get bad TF-IDF neighbours?

Before running aggregate metrics, let's **diagnose** the specific failure.

In [8]:
name = "Santorini"
qi   = IDX[name]
q    = PLACES[qi]

# Raw cosine similarities from TF-IDF to every other place
sims_sparse = sparse_embs @ sparse_embs[qi]
sims_dense  = dense_embs  @ dense_embs[qi]

print(f"Cosine similarity of '{name}' to every other place\n")
print(f"{'Place':<30} {'TF-IDF sim':>12} {'Dense sim':>12} {'Category':<12} {'Relevant?'}")
print("-" * 80)

rows = sorted(
    [(sims_sparse[i], sims_dense[i], PLACES[i]) for i in range(N) if i != qi],
    key=lambda x: -x[1]   # sort by dense sim to see what's truly similar
)
for ss, sd, p in rows:
    rel = label_relevance(q, p)
    flag = "✓" if rel >= 1.0 else ("~" if rel > 0 else "")
    print(f"{p['name']:<30} {ss:>12.4f} {sd:>12.4f} {p['category']:<12} {flag}")

print(f"\nKey insight: TF-IDF similarities for '{name}' are very close to 0.")
print(f"  Max TF-IDF sim : {max(sims_sparse[i] for i in range(N) if i != qi):.4f}")
print(f"  Max Dense sim  : {max(sims_dense[i]  for i in range(N) if i != qi):.4f}")
print(f"  → TF-IDF vectors barely overlap → neighbour selection is essentially random.")

Cosine similarity of 'Santorini' to every other place

Place                            TF-IDF sim    Dense sim Category     Relevant?
--------------------------------------------------------------------------------
Meteora                              0.0443       0.5508 heritage     ~
Crete                                0.0724       0.5486 nature       ~
Mykonos                              0.0803       0.5371 beach        ✓
French Riviera                       0.0541       0.4604 beach        ✓
Amalfi Coast                         0.0299       0.4342 beach        ✓
Ibiza                                0.0305       0.3947 beach        ✓
Acropolis of Athens                  0.0154       0.3814 heritage     ~
Plitvice Lakes                       0.0077       0.3590 nature       
Mont Saint-Michel                    0.0158       0.3562 heritage     
Pompeii                              0.0388       0.3483 heritage     
Sintra Palaces                       0.0079       0.3412 heritage  

---
## 5. Full evaluation across all places

For each place as a query, retrieve top-K, then score against **label-based** ground truth.

Systems compared:
- `Sparse oracle` — exact brute-force over TF-IDF (ceiling for HNSW sparse)
- `Dense oracle` — exact brute-force over sentence embeddings (ceiling for HNSW dense)
- `HNSW sparse` — approximate search over TF-IDF
- `HNSW dense` — approximate search over sentence embeddings

In [9]:
def evaluate_all(k: int = 5) -> None:
    results = defaultdict(lambda: defaultdict(list))  # system → metric → [scores]

    for qi, q in enumerate(PLACES):
        # Ground truth (label-based)
        gt_ids     = set(label_ground_truth(q, k=k*3))  # fetch more than k for completeness
        rel_scores = {i: label_relevance(q, PLACES[i]) for i in range(N) if i != qi}

        if not gt_ids:
            continue   # skip places with no relevant neighbors (shouldn't happen)

        systems = {
            "Sparse oracle" : oracle_top_k(sparse_embs, qi, k),
            "Dense oracle"  : oracle_top_k(dense_embs,  qi, k),
            "HNSW sparse"   : [IDX[p['name']] for _, p in sparse_idx.search(sparse_embs[qi], k=k)],
            "HNSW dense"    : [IDX[p['name']] for _, p in dense_idx.search(dense_embs[qi],   k=k)],
        }

        for sys_name, ret_ids in systems.items():
            results[sys_name]["Recall@K"].append(recall_at_k(ret_ids,    gt_ids,     k))
            results[sys_name]["P@K"].append(      precision_at_k(ret_ids, gt_ids,     k))
            results[sys_name]["MRR"].append(      mrr(ret_ids,            gt_ids))
            results[sys_name]["NDCG@K"].append(   ndcg_at_k(ret_ids,     rel_scores, k))

    print(f"Evaluation @K={k}  |  Ground truth: label-based  |  N={N} queries\n")
    print(f"{'System':<18} {'Recall@K':>10} {'P@K':>8} {'MRR':>8} {'NDCG@K':>10}")
    print("-" * 60)
    for sys_name in ["Sparse oracle", "HNSW sparse", "Dense oracle", "HNSW dense"]:
        m = results[sys_name]
        print(
            f"{sys_name:<18}"
            f" {np.mean(m['Recall@K']):>10.3f}"
            f" {np.mean(m['P@K']):>8.3f}"
            f" {np.mean(m['MRR']):>8.3f}"
            f" {np.mean(m['NDCG@K']):>10.3f}"
        )
    print()
    print("Legend:")
    print("  Sparse oracle  = TF-IDF ceiling (what perfect HNSW on TF-IDF could achieve)")
    print("  Dense oracle   = SentTrans ceiling (what perfect HNSW on dense could achieve)")
    print("  HNSW sparse    = TF-IDF with approximate graph search")
    print("  HNSW dense     = SentTrans with approximate graph search")


for k in [3, 5, 10]:
    evaluate_all(k)
    print()

Evaluation @K=3  |  Ground truth: label-based  |  N=47 queries

System               Recall@K      P@K      MRR     NDCG@K
------------------------------------------------------------
Sparse oracle           0.532    0.532    0.762      0.687
HNSW sparse             0.333    0.333    0.443      0.400
Dense oracle            0.468    0.468    0.706      0.762
HNSW dense              0.305    0.305    0.333      0.407

Legend:
  Sparse oracle  = TF-IDF ceiling (what perfect HNSW on TF-IDF could achieve)
  Dense oracle   = SentTrans ceiling (what perfect HNSW on dense could achieve)
  HNSW sparse    = TF-IDF with approximate graph search
  HNSW dense     = SentTrans with approximate graph search

Evaluation @K=5  |  Ground truth: label-based  |  N=47 queries

System               Recall@K      P@K      MRR     NDCG@K
------------------------------------------------------------
Sparse oracle           0.557    0.557    0.844      0.654
HNSW sparse             0.396    0.396    0.557      0

---
## 6. Per-category breakdown

Where is each embedding type strongest? A category breakdown reveals which *types* of travel queries each system handles well.

In [10]:
K = 5
categories = sorted(set(p['category'] for p in PLACES))

print(f"Recall@{K} by category  (ground truth = label-based)\n")
print(f"{'Category':<12} {'N':>4}  {'Sparse oracle':>14} {'HNSW sparse':>12} {'Dense oracle':>13} {'HNSW dense':>11}")
print("-" * 70)

for cat in categories:
    cat_places = [(qi, p) for qi, p in enumerate(PLACES) if p['category'] == cat]
    row = {s: [] for s in ["Sparse oracle", "HNSW sparse", "Dense oracle", "HNSW dense"]}

    for qi, q in cat_places:
        gt_ids = set(label_ground_truth(q, k=K*3))
        if not gt_ids:
            continue
        systems = {
            "Sparse oracle": oracle_top_k(sparse_embs, qi, K),
            "HNSW sparse"  : [IDX[p['name']] for _, p in sparse_idx.search(sparse_embs[qi], k=K)],
            "Dense oracle" : oracle_top_k(dense_embs,  qi, K),
            "HNSW dense"   : [IDX[p['name']] for _, p in dense_idx.search(dense_embs[qi],  k=K)],
        }
        for sys, ret in systems.items():
            row[sys].append(recall_at_k(ret, gt_ids, K))

    n = len(cat_places)
    so = np.mean(row['Sparse oracle']) if row['Sparse oracle'] else 0
    hs = np.mean(row['HNSW sparse'])   if row['HNSW sparse']   else 0
    do = np.mean(row['Dense oracle'])  if row['Dense oracle']  else 0
    hd = np.mean(row['HNSW dense'])    if row['HNSW dense']    else 0
    print(f"{cat:<12} {n:>4}  {so:>14.3f} {hs:>12.3f} {do:>13.3f} {hd:>11.3f}")

Recall@5 by category  (ground truth = label-based)

Category        N   Sparse oracle  HNSW sparse  Dense oracle  HNSW dense
----------------------------------------------------------------------
beach           7           0.514        0.286         0.743       0.429
heritage       16           0.625        0.463         0.625       0.588
landmark        8           0.375        0.350         0.250       0.100
nature         16           0.600        0.400         0.637       0.500


---
## 7. HNSW approximation gap

**Approximation gap** = oracle recall − HNSW recall.
This measures how much the graph structure (not the embedding) is hurting you.

In [11]:
K = 5
sparse_oracle_r, sparse_hnsw_r = [], []
dense_oracle_r,  dense_hnsw_r  = [], []

for qi, q in enumerate(PLACES):
    gt_ids = set(label_ground_truth(q, k=K*3))
    if not gt_ids:
        continue
    sparse_oracle_r.append(recall_at_k(oracle_top_k(sparse_embs, qi, K), gt_ids, K))
    sparse_hnsw_r.append(  recall_at_k([IDX[p['name']] for _, p in sparse_idx.search(sparse_embs[qi], K)], gt_ids, K))
    dense_oracle_r.append( recall_at_k(oracle_top_k(dense_embs,  qi, K), gt_ids, K))
    dense_hnsw_r.append(   recall_at_k([IDX[p['name']] for _, p in dense_idx.search(dense_embs[qi],  K)], gt_ids, K))

print(f"Recall@{K} breakdown  (label ground truth)\n")
print(f"{'System':<18} {'Oracle':>8} {'HNSW':>8} {'Gap':>8}  Interpretation")
print("-" * 75)

so, sh = np.mean(sparse_oracle_r), np.mean(sparse_hnsw_r)
_do, dh = np.mean(dense_oracle_r),  np.mean(dense_hnsw_r)

print(f"{'TF-IDF':<18} {so:>8.3f} {sh:>8.3f} {so-sh:>8.3f}  ← embedding quality ceiling")
print(f"{'Sentence-Transformers':<18} {_do:>8.3f} {dh:>8.3f} {_do-dh:>8.3f}  ← HNSW graph quality")
print()
print("Interpretation:")
print("  Oracle score  — how good the *embedding* is at capturing category similarity")
print("  HNSW score    — how well the *graph* finds what the embedding considers similar")
print("  Gap           — cost of approximate search; should be small (<0.05) on this dataset")
print()
if so < 0.4:
    print("⚠ TF-IDF oracle recall is LOW → the embedding itself is the problem, not the graph.")
    print("  Sparse TF-IDF vectors have low cosine overlap between semantically similar places.")
    print("  Sentence embeddings (dense) should show a much higher oracle recall.")

Recall@5 breakdown  (label ground truth)

System               Oracle     HNSW      Gap  Interpretation
---------------------------------------------------------------------------
TF-IDF                0.557    0.396    0.162  ← embedding quality ceiling
Sentence-Transformers    0.583    0.451    0.132  ← HNSW graph quality

Interpretation:
  Oracle score  — how good the *embedding* is at capturing category similarity
  HNSW score    — how well the *graph* finds what the embedding considers similar
  Gap           — cost of approximate search; should be small (<0.05) on this dataset



---
## 8. Summary — what the numbers tell you

Run this cell after the evaluation cells above to get a plain-English verdict.

In [12]:
print("=" * 65)
print("VERDICT")
print("=" * 65)

gap_threshold  = 0.05   # acceptable HNSW approximation loss
good_threshold = 0.60   # a retrieval system we'd consider 'good'

lines = [
    ("TF-IDF embedding quality",
     so,
     "✅ Good" if so >= good_threshold else "❌ Poor — TF-IDF fails to capture semantic similarity."),

    ("Dense embedding quality",
     _do,
     "✅ Good" if _do >= good_threshold else "⚠ Moderate — sentence model may benefit from fine-tuning."),

    ("HNSW graph (sparse) gap",
     so - sh,
     "✅ Small — graph navigates TF-IDF space well." if so - sh <= gap_threshold
     else "⚠ Large — increase ef_construction or M to improve graph quality."),

    ("HNSW graph (dense) gap",
     _do - dh,
     "✅ Small — graph navigates dense space well." if _do - dh <= gap_threshold
     else "⚠ Large — increase ef_construction or M."),
]

for label, val, verdict in lines:
    print(f"  {label:<30} {val:>6.3f}  {verdict}")

print()
print("Bottom line:")
print("  The bad neighbours you saw (Santorini → Eiffel Tower) come from")
print("  TF-IDF cosine similarities that are nearly ZERO between all places.")
print("  When all similarities are ~0, HNSW graph wiring is essentially random.")
print("  Dense embeddings understand 'beach island Greece' ≈ 'turquoise Mediterranean cove'.")
print("  Use hybrid retrieval (v2 notebook) to get the best of both.")

VERDICT
  TF-IDF embedding quality        0.557  ❌ Poor — TF-IDF fails to capture semantic similarity.
  Dense embedding quality         0.583  ⚠ Moderate — sentence model may benefit from fine-tuning.
  HNSW graph (sparse) gap         0.162  ⚠ Large — increase ef_construction or M to improve graph quality.
  HNSW graph (dense) gap          0.132  ⚠ Large — increase ef_construction or M.

Bottom line:
  The bad neighbours you saw (Santorini → Eiffel Tower) come from
  TF-IDF cosine similarities that are nearly ZERO between all places.
  When all similarities are ~0, HNSW graph wiring is essentially random.
  Dense embeddings understand 'beach island Greece' ≈ 'turquoise Mediterranean cove'.
  Use hybrid retrieval (v2 notebook) to get the best of both.
